# ANN RegressionPredicting a continuous numeric target with an Artificial Neural Network — California Housing dataset. From-scratch NumPy implementation, then Keras verification.

## Task and Dataset**Goal:** predict median house value for California districts from features like median income, house age, average rooms, population, and location — a regression problem.**Dataset:** California Housing dataset (built into scikit-learn).**Quick refresher — why this is a regression ANN:**- Output layer has **1 neuron, no activation (linear)** — regression outputs are unbounded- Loss function: **Mean Squared Error (MSE)**- Hidden layers use **ReLU**

In [ ]:
from sklearn.datasets import fetch_california_housingimport numpy as npimport pandas as pddata = fetch_california_housing()X = data.datay = data.target.reshape(-1, 1)df = pd.DataFrame(X, columns=data.feature_names)df["MedHouseVal"] = yprint(df.shape)df.head()

## Preprocessing- Scale features (regression is sensitive to feature scale, since large-magnitude features dominate the loss gradient)- Scale the target too — keeps gradients well-behaved during manual training- Train/test split

In [ ]:
from sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitX_scaler = StandardScaler()X_scaled = X_scaler.fit_transform(X)y_scaler = StandardScaler()y_scaled = y_scaler.fit_transform(y)X_train, X_test, y_train, y_test = train_test_split(    X_scaled, y_scaled, test_size=0.2, random_state=42)print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## Concept Refresher: Regression ANN$$z^{[1]} = W^{[1]}X + b^{[1]}, \quad a^{[1]} = ReLU(z^{[1]})$$$$z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}, \quad a^{[2]} = ReLU(z^{[2]})$$$$\hat y = W^{[3]}a^{[2]} + b^{[3]} \quad \text{(linear output, no activation)}$$Loss (MSE):$$L = \frac{1}{n}\sum (y - \hat y)^2$$The only structural differences from the classification ANN: no activation on the output layer, and MSE instead of binary crossentropy.

In [ ]:
def relu(z):    return np.maximum(0, z)def relu_derivative(z):    return (z > 0).astype(float)def mse_loss(y_true, y_pred):    return np.mean((y_true - y_pred) ** 2)

In [ ]:
np.random.seed(42)n_features = X_train.shape[1]h1_size = 16h2_size = 8output_size = 1W1 = np.random.randn(n_features, h1_size) * np.sqrt(2 / n_features)b1 = np.zeros((1, h1_size))W2 = np.random.randn(h1_size, h2_size) * np.sqrt(2 / h1_size)b2 = np.zeros((1, h2_size))W3 = np.random.randn(h2_size, output_size) * np.sqrt(2 / h2_size)b3 = np.zeros((1, output_size))lr = 0.01epochs = 300batch_size = 64n = X_train.shape[0]losses = []for epoch in range(epochs):    idx = np.random.permutation(n)    X_shuf, y_shuf = X_train[idx], y_train[idx]    for start in range(0, n, batch_size):        end = start + batch_size        Xb, yb = X_shuf[start:end], y_shuf[start:end]        # Forward        z1 = Xb @ W1 + b1        a1 = relu(z1)        z2 = a1 @ W2 + b2        a2 = relu(z2)        y_pred = a2 @ W3 + b3   # linear output        # Backward        d3 = 2 * (y_pred - yb) / len(Xb)        dW3 = a2.T @ d3        db3 = np.sum(d3, axis=0, keepdims=True)        d2 = (d3 @ W3.T) * relu_derivative(z2)        dW2 = a1.T @ d2        db2 = np.sum(d2, axis=0, keepdims=True)        d1 = (d2 @ W2.T) * relu_derivative(z1)        dW1 = Xb.T @ d1        db1 = np.sum(d1, axis=0, keepdims=True)        W3 -= lr * dW3; b3 -= lr * db3        W2 -= lr * dW2; b2 -= lr * db2        W1 -= lr * dW1; b1 -= lr * db1    # epoch loss on full train set    a1_full = relu(X_train @ W1 + b1)    a2_full = relu(a1_full @ W2 + b2)    y_pred_full = a2_full @ W3 + b3    epoch_loss = mse_loss(y_train, y_pred_full)    losses.append(epoch_loss)    if epoch % 30 == 0:        print(f"Epoch {epoch}, Loss: {epoch_loss:.4f}")

In [ ]:
import matplotlib.pyplot as pltplt.plot(losses)plt.title("Manual NumPy ANN - Training Loss")plt.xlabel("Epoch")plt.ylabel("MSE Loss (scaled target)")plt.grid(True)plt.show()# Test performancea1_test = relu(X_test @ W1 + b1)a2_test = relu(a1_test @ W2 + b2)y_pred_test = a2_test @ W3 + b3test_mse = mse_loss(y_test, y_pred_test)print("Manual NumPy Test MSE (scaled):", test_mse)

## Keras VerificationSame architecture — Dense(16, relu) -> Dense(8, relu) -> Dense(1, linear) — built with Keras.

In [ ]:
import tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersmodel = keras.Sequential([    layers.Dense(16, activation='relu', input_shape=(n_features,)),    layers.Dense(8, activation='relu'),    layers.Dense(1)   # linear output for regression])model.compile(optimizer='adam', loss='mse')history = model.fit(X_train, y_train, epochs=30, batch_size=64,                     validation_split=0.1, verbose=0)test_loss = model.evaluate(X_test, y_test, verbose=0)print("Keras Test MSE (scaled):", test_loss)

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')plt.plot(history.history['val_loss'], label='Val Loss')plt.title("Keras ANN - Training vs Validation Loss")plt.xlabel("Epoch")plt.ylabel("MSE")plt.legend()plt.grid(True)plt.show()

## ConclusionThe regression ANN differs from the classification one in just two places: a linear (no activation) output neuron, and MSE loss instead of binary crossentropy. Both the manual and Keras versions converge to a similar MSE on the scaled target, confirming the from-scratch implementation is correct.